## Import Dependencies

In [1]:
import pandas as pd
import numpy as np
import xlsxwriter
import math
import schwabdev
from scipy import stats

## Getting S&P 500 Constituents

In [2]:
sp_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
sp500_constituents = pd.read_html(sp_url, header=0)[0]

In [3]:
sp500_constituents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


In [4]:
sp500_constituents = sp500_constituents['Symbol'].to_list()

### Changing format to match Schwab's

In [5]:
sp500_constituents = [x.replace('.', '/') for x in sp500_constituents]

## Setting up Schwab API

In [6]:
from key import api_key, api_secret

In [8]:
client = schwabdev.Client(app_key=api_key, app_secret=api_secret, capture_callback=False)

# Value Investing Strategy 

We will look at a composite score of the following metrics:

- Price-to-earnings ratio - allows us to check if stock is over/under - valued
- Price-to-book ratio
- Price-to-earnings-growth
- Price-to-cash-flow ratio 


## Making API Calls

In [26]:
df_columns = ['Stock', 'Price', 'PE Ratio', 'PE Percentile', 'PB Ratio', 'PB Percentile', 'PEG Ratio', 'PEG Perntile', 'PCF Ratio', 'PCF Percentile']

In [15]:
data = client.instruments('AAPL', projection='fundamental').json()

In [44]:
price = client.quote('BRK/B').json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [39]:
print(price)

211.46


In [21]:
print(data['instruments'][0])

{'fundamental': {'symbol': 'AAPL', 'high52': 260.1, 'low52': 169.11, 'dividendAmount': 1.0, 'dividendYield': 0.47783, 'dividendDate': '2025-02-10 00:00:00.0', 'peRatio': 33.46461, 'pegRatio': -42.19818, 'pbRatio': 57.58502, 'prRatio': 9.84022, 'pcfRatio': 30.49799, 'grossMarginTTM': 46.5188, 'grossMarginMRQ': 46.8825, 'netProfitMarginTTM': 24.295, 'netProfitMarginMRQ': 29.2276, 'operatingMarginTTM': 24.295, 'operatingMarginMRQ': 29.2276, 'returnOnEquity': 136.5204, 'returnOnAssets': 22.5192, 'returnOnInvestment': 48.1424, 'quickRatio': 0.7833, 'currentRatio': 0.92294, 'interestCoverage': 0.0, 'totalDebtToCapital': 51.3313, 'ltDebtToEquity': 144.9998, 'totalDebtToEquity': 125.7617, 'epsTTM': 6.27947, 'epsChangePercentTTM': -2.1727, 'epsChangeYear': 0.0, 'epsChange': 0.0, 'revChangeYear': 2.0219, 'revChangeTTM': 2.6066, 'revChangeIn': 0.0, 'sharesOutstanding': 15022073000.0, 'marketCapFloat': 0.0, 'marketCap': 3143819437440.0, 'bookValuePerShare': 4.43848, 'shortIntToFloat': 0.0, 'shortI

In [42]:
entries = []
for stock in sp500_constituents:
    print(stock)
    data = client.instruments(stock, projection='fundamental').json()
    price = client.quote(stock).json()[stock]['quote']['lastPrice']
    pe_ratio = data['instruments'][0]['fundamental']['peRatio']
    peg_ratio = data['instruments'][0]['fundamental']['pegRatio']
    pb_ratio = data['instruments'][0]['fundamental']['pbRatio']
    pcf_ratio = data['instruments'][0]['fundamental']['pcfRatio']
    entries.append(
        [
            stock,
            price,
            pe_ratio,
            'N/a',
            pb_ratio,
            'N/a',
            peg_ratio,
            'N/a',
            pcf_ratio,
            'N/a'
        ]
    )

MMM
AOS
ABT
ABBV
ACN
ADBE
AMD
AES
AFL
A
APD
ABNB
AKAM
ALB
ARE
ALGN
ALLE
LNT
ALL
GOOGL
GOOG
MO
AMZN
AMCR
AEE
AEP
AXP
AIG
AMT
AWK
AMP
AME
AMGN
APH
ADI
ANSS
AON
APA
APO
AAPL
AMAT
APTV
ACGL
ADM
ANET
AJG
AIZ
T
ATO
ADSK
ADP
AZO
AVB
AVY
AXON
BKR
BALL
BAC
BAX
BDX
BRK/B


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [40]:
value_df = pd.DataFrame(entries, columns = df_columns)

In [41]:
value_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Stock           60 non-null     object 
 1   Price           60 non-null     float64
 2   PE Ratio        60 non-null     float64
 3   PE Percentile   60 non-null     object 
 4   PB Ratio        60 non-null     float64
 5   PB Percentile   60 non-null     object 
 6   PEG Ratio       60 non-null     float64
 7   PEG Perntile    60 non-null     object 
 8   PCF Ratio       60 non-null     float64
 9   PCF Percentile  60 non-null     object 
dtypes: float64(5), object(5)
memory usage: 4.8+ KB
